# Parse House Roll Call PDF files

This notebook parses and saves data to CSV files from the House Roll Call vote PDFs that are stored in ```../../data/HJRollCalls'``` subfolders.

## Parsing a Single PDF File

First, I will parse the first House Roll Call Vote pdf in our dataset (stored at ```../../data/HJRollCalls/2003/hj01082003.pdf```)

This approach uses PyMuPDF to extract text from the PDF by line, then applies regular expressions to appropriately capture the data. This method works well because we know that the structure of the House Roll Call Vote Journals is consistent across the years, so regular expressions can capture the names of the different representatives as long as they are robust enough to handle some varied formatting.

### Setup

In [1]:
import pymupdf
import re
import pandas as pd
from pathlib import Path
from datetime import datetime
from tqdm import tqdm

Regular expressions needed to parse data are defined.

Depending on the year of the document, the required Regular expressions vary.

In [2]:
# Define regular expressions for validating a file's content
pre2009_patterns_to_match =[
    '^\s+MASSACHUSETTS HOUSE OF REPRESENTATIVES\s+$',
    '^\s+Yea and Nay\s+(?P<RCDate>\d{2}/\d{2}/\d{2})\s(?P<RCTime>\d{2}:\d{2})\s\D$',
    '^\s+No.\s+(?P<RCNumberForYear>\d+)\s+(?P<YeaVoteCount>\d+)\s+YEAS\s+(?P<NayVoteCount>\d+)\s+NAYS\s+(?P<NotVotingCount>\d+)\s+N/V\s+$',
    '^\s+(?P<Vote1>[Y,N,X,P])\s+(?P<Name1>Mr. Speaker|(.+?)(?:\s{,1}[A-Za-z.,\'-]+)+)\s+(?P<Vote2>[Y,N,X,P])\s+(?P<Name2>(.+?)(?:\s{,1}[A-Za-z.,\'-]+)+)\s+(?P<Vote3>[Y,N,X,P])\s+(?P<Name3>(.+?)(?:\s{,1}[A-Za-z.,\'-]+)+)\s+((?P<Vote4>[Y,N,X,P])\s+(?P<Name4>(.+?)(?:\s{,1}[A-Za-z.,\'-]+)+)\s+)?$',
    '^\s+\*=AFTER VOTE\s+P=PRESENT\s+X=NOT VOTING\s+$'
]

In [3]:
post2009pre2017_patterns_to_match = [
    '^YEAS: (?P<YeaVoteCount>\d+).?$',
    '^NAYS: (?P<NayVoteCount>\d+).?$',
    '^N-V: (?P<NVVoteCount>\d+).?$',
    '^P=PRESENT\s?$',
    '^X=NOT VOTING \s?$',
    '^\*=AFTER VOTE$',
    '^Yea and Nay Supplement$',
    '^JOURNAL OF THE HOUSE\.?$',
    '^(‘)?\D+,\s(?P<VoteDate>\D+\d{1,2},\s\d{4}).*$',
    '^Yea and Nay No\. (?P<RollCallCount>\d+).?$',
    '^(?P<VoteValue>[YNPXynpx]{1}).?$',
    '^(?P<Name>\*?([A-Za-z][A-ZÀ-ÿa-z.,\ \'’-]+?)(?: {,1}[\*À-ÿA-Za-z.,\'’-]+)+).?$',
    '^YEAS: (?P<YeaVoteCount>\d+)\s+NAYS: (?P<NayVoteCount>\d+).?$',
    '^P=PRESENT        X=NOT VOTING      \*=AFTER VOTE$'
]

In [4]:
post2017_patterns_to_match = [
    '^(?P<RCDate>\d{2}/\d{2}/\d{4})\s(?P<RCTime>\d{2}:\d{2} \D{2})$',
    '^\s+MASSACHUSETTS HOUSE OF REPRESENTATIVES$',
    '^(?P<VoteCount>\d+)\s(?P<VoteType>\D+)$',
    '^No.\s+(?P<RCNumberForYear>\d+)$',
    '^Yea and Nay$',
    '^(?P<Name>Mr. Speaker|([A-Z-][À-ÿa-z.,\'’-]+?)(?: {,1}[\*À-ÿA-Za-z.,\'’-]+)+)$',
    '^(?P<VoteValue>\D{1})$',
    '^\*=AFTER VOTE\s+P=PRESENT\s+X=NOT VOTING$',
    '^(?P<VoteValue>[YNPXynpx]{1})$',
    '^(?P<Name>Mr Speaker|([A-Z-][À-ÿa-z.,\'’-]+?)(?: {,1}[\*À-ÿA-Za-z.,\'’-]+)+).?$'
]

There are a few files that specifically do not match the patterns seen in many of the others. It appears that although the document appears the same way compared to files from after 2017, the files from 2023 and onwards fix some of the encoding issues that resulted in Nay votes appearing out of order and the main title and other features of the document not appearing in expected order.

While a more robust solution can be expanded (or split into a separate function) for simplicity for now, the functionality is being controlled by tracking the years to except and will be updated in the future during clean-up phase of this project

In [5]:
years_to_except = [2023, 2024, 2025]

In [6]:
years_to_court_number = {
    2003: 182,
    2004: 183,
    2005: 183,
    2006: 184,
    2007: 184,
    2008: 185,
    2009: 185,
    2010: 186,
    2011: 187,
    2012: 187,
    2013: 188,
    2014: 188,
    2015: 189,
    2016: 189,
    2017: 190,
    2018: 190,
    2019: 191,
    2020: 191,
    2021: 192,
    2022: 192,
    2023: 193,
    2024: 193,
    2025: 194,
}

Parsing functions for the different years of PDFs are defined. Currently PDFs from before 2009 and after 2017 are supported and are defined in the following two functions.

In [7]:
def parse_Pre2009_PDF(path, bill_id, court_number, verbose=False):

    doc = pymupdf.open(path)
    all_pdf_bills = []
    all_pdf_votes = []

    for page in doc:
        text = page.get_text()
        lines = text.split('\n')

        current_pdf_bill_data = []
        current_pdf_bill_data.append(bill_id)
        current_pdf_bill_data.append(court_number)

        rc_count_for_year = 0

        current_pdf_vote_data = []

        for i, line in enumerate(lines):
            if verbose: print(f'{i}: {line}')

            # Skip parsing if blank line
            if line.isspace():
                if verbose: print("Found blank line")
                continue

            if i == 2: # parse full title line individually for ease
                current_pdf_bill_data.append(line.strip())

            # match = re.search(title_regex, line)
            matched = False
            for j, pattern in enumerate(pre2009_patterns_to_match):
                match = re.match(pattern, line)

                if match:
                    matched = True
                    if verbose: print(f'Matched with pattern {pattern}')

                    if j == 0:
                        # Do nothing, just the title
                        if verbose: print("Parsing title line")
                    elif j == 1:
                        # Extract vote date and time
                        rc_date = match.group('RCDate')
                        rc_date = datetime.strptime(rc_date, "%m/%d/%y").strftime("%m/%d/%Y")
                        current_pdf_bill_data.append(rc_date)
                        rc_time = match.group('RCTime')
                        if int(rc_time[:2]) < 12 and int(rc_time[:2]) >= 11:
                            rc_time += ' AM'
                        else: 
                            rc_time += ' PM'
                        current_pdf_bill_data.append(rc_time)
                        if verbose: print(f"Found date: {rc_date} and time: {rc_time}")
                    elif j == 2:
                        rc_count_for_year = match.group('RCNumberForYear')
                        current_pdf_bill_data.append(rc_count_for_year)

                        rc_yea_vote_count = match.group('YeaVoteCount')
                        current_pdf_bill_data.append(rc_yea_vote_count)

                        rc_nay_vote_count = match.group('NayVoteCount')
                        current_pdf_bill_data.append(rc_nay_vote_count)

                        rc_not_voting_count = match.group('NotVotingCount')
                        current_pdf_bill_data.append(rc_not_voting_count)

                        if verbose: print(f'Found {rc_yea_vote_count} YEA votes, {rc_nay_vote_count} Nay votes, {rc_not_voting_count} NV votes for RC Vote {rc_count_for_year}')
                    elif j == 3:
                        vote1 = match.group('Vote1')
                        name1 = match.group('Name1')
                        current_pdf_vote_data.append([court_number, rc_count_for_year, name1, vote1])

                        vote2 = match.group('Vote2')
                        name2 = match.group('Name2')
                        current_pdf_vote_data.append([court_number, rc_count_for_year, name2, vote2])

                        vote3 = match.group('Vote3')
                        name3 = match.group('Name3')
                        current_pdf_vote_data.append([court_number, rc_count_for_year, name3, vote3])

                        vote4 = match.group('Vote4')
                        name4 = match.group('Name4')
                        if not vote4 == None: current_pdf_vote_data.append([court_number, rc_count_for_year, name4, vote4])

                        if verbose: print(f'Found {name1} with vote {vote1}, {name2} with vote {vote2}, {name3} with {vote3}')
                    elif j == 4:
                        if verbose: print("Parsing end line")

            if not matched:
                Exception(f"Failed to match regex on line {i} : {line}")

        all_pdf_bills.append(current_pdf_bill_data)
        all_pdf_votes.extend(current_pdf_vote_data)
        bill_id += 1

    new_bill_id = bill_id
    print(f"Succesfully parsed file at path: {path}")
    if verbose:
        print(f'Found {len(all_pdf_votes)} voting records')
        print(f'Found {len(all_pdf_bills)} bill records')
    return all_pdf_bills, all_pdf_votes, new_bill_id

In [8]:
def parse_2009through2017_PDF(path, bill_id, court_number, verbose=False):
    doc = pymupdf.open(path)
    all_pdf_bills = []
    all_pdf_votes = []

    for page_count, page in enumerate(doc):
        print(f"PAGE #{page_count + 1}")
        text = page.get_text()
        lines = text.split('\n')
        rc_count_in_year = 0

        current_pdf_bill_data = []
        current_pdf_vote_data = []
        current_pdf_bill_data.append(bill_id)
        current_pdf_bill_data.append(court_number)

        bill_name_parsed = False
        skip_next_line = False  # Flag to skip name line after parsing vote
        rc_count = None  # Store roll call count for vote records

        for i, line in enumerate(lines):
            if verbose: print(f'{i}: {line}')

            # Skip parsing if blank line
            if line.isspace() or not line:
                if verbose: print("Found blank line")
                continue

            # Skip this line if it was already processed as a name following a vote
            if skip_next_line:
                skip_next_line = False
                continue

            matched = False

            for j, pattern in enumerate(post2009pre2017_patterns_to_match):
                match = re.match(pattern, line)

                if match:
                    matched = True
                    if verbose: print(f'Matched with pattern {pattern}')

                    # Parse Yay vote count
                    if j == 0:
                        yay_vote_count = int(match.group('YeaVoteCount'))
                        current_pdf_bill_data.append(yay_vote_count)
                        if verbose: print(f'Found {yay_vote_count} Yay votes')
                    elif j == 1:
                        nay_vote_count = int(match.group('NayVoteCount'))
                        current_pdf_bill_data.append(nay_vote_count)
                        if verbose: print(f'Found {nay_vote_count} Nay votes')
                    elif j == 2:
                        nv_vote_count = int(match.group('NVVoteCount'))
                        current_pdf_bill_data.append(nv_vote_count)
                        if verbose: print(f'Found {nv_vote_count} NV votes')
                    elif j == 3:
                        if verbose: print("Parsing Present text line")
                    elif j == 4:
                        if verbose: print("Parsing Not voting text line")
                    elif j == 5:
                        if verbose: print("Parsing After vote  text line")
                    elif j == 6:
                        if verbose: print("Parsing Yay Nay supplement text line")
                        continue
                    elif j == 7:
                        if verbose: print("Parsing JOURNAL OF THE HOUSE text line")
                    elif j == 8:
                        # Parse date

                        # Extract vote date and convert to string
                        date_string = match.group('VoteDate')

                        if 'Ocotober' in date_string:
                            date_string = date_string.replace('Ocotober', 'October')
                        
                        numeric_date_string = ''

                        try:

                            # Define the format of your date string
                            date_format = "%B %d, %Y"

                            # Convert the string to a datetime object
                            datetime_object = datetime.strptime(date_string, date_format)

                            # Format the datetime object into a numerical string (e.g., YYYYMMDD)
                            numeric_date_string = datetime_object.strftime("%m/%d/%Y")
                        except:
                            print("failed to convert date string")
                            numeric_date_string = '01/01/1900' # set invalid string to old value
                        finally:
                            current_pdf_bill_data.append(numeric_date_string)

                            # Append a null value for the RC time for bills from this year
                            current_pdf_bill_data.append(None)

                            if verbose: print(f"Found date: {numeric_date_string}")
                    elif j == 9:
                        # Extract roll call count
                        rc_count_in_year = int(match.group('RollCallCount'))
                        current_pdf_bill_data.append(rc_count_in_year)

                        if verbose: print(f"Found Roll Call Count: {rc_count_in_year}")
                    elif j == 10:
                        # Parse vote value - check if next line exists and is a name
                        vote = match.group('VoteValue').strip().upper()
                        
                        # Check if next line exists and is within bounds
                        if i + 1 < len(lines):
                            name_text = lines[i + 1].strip()
                            # Remove digits that might appear in the name line
                            name_text = re.sub(r'\d+', '', name_text)
                            
                            if verbose: print(f'Peeking next line: {name_text}')
                            
                            # Try to match name pattern
                            name_match = re.match(post2009pre2017_patterns_to_match[11], name_text)
                            if name_match:
                                name = name_match.group('Name').replace('.', '').strip()
                                current_pdf_vote_data.append([court_number, rc_count_in_year, name, vote])
                                skip_next_line = True  # Skip the name line on next iteration
                                if verbose: print(f'Found vote {vote} from Rep. {name}')
                            else:
                                # Name might be on same line or combined - try to extract
                                if verbose: print(f'Could not parse name from next line, vote: {vote}')
                        else:
                            if verbose: print(f'No next line available for vote {vote}')
                    elif j == 11:
                        # This is a name line - check if it might be combined with a vote
                        # Try to see if there's a vote value at the start or end
                        name_text = line.strip()
                        name_text = re.sub(r'\d+', '', name_text)
                        
                        # Check if previous line was a vote (should have been handled, but check anyway)
                        if i > 0:
                            prev_line = lines[i-1].strip()
                            vote_match = re.match(r'^(?P<VoteValue>[YNPXynpx]{1})$', prev_line)
                            if vote_match:
                                # Previous line was a vote, this is the name
                                vote = vote_match.group('VoteValue').upper()
                                name = match.group('Name').replace('.', '').strip()
                                current_pdf_vote_data.append([court_number, rc_count_in_year, name, vote])
                                if verbose: print(f'Found vote {vote} from Rep. {name} (from previous line)')
                            else:
                                # Check if vote and name are on same line (e.g., "N Name" or "Name N")
                                combined_match = re.match(r'^(?P<VoteValue>[YNPXynpx]{1})\s+(?P<Name>.+)$', line)
                                if combined_match:
                                    vote = combined_match.group('VoteValue').upper()
                                    name = combined_match.group('Name').strip()
                                    name = re.sub(r'\d+', '', name)
                                    current_pdf_vote_data.append([court_number, rc_count_in_year, name, vote])
                                    if verbose: print(f'Found combined vote {vote} and name {name}')
                                else:
                                    if verbose: print("Found person name but skipping because no associated vote found")
                        else:
                            if verbose: print("Found person name but skipping because is already parsed or no vote")
                    elif j == 12:
                        # Parse yay and nay on same line
                        yay_vote_count = int(match.group('YeaVoteCount'))
                        nay_vote_count = int(match.group('NayVoteCount'))
                        current_pdf_bill_data.append(yay_vote_count)
                        current_pdf_bill_data.append(nay_vote_count)

                        if verbose: print(f'Found {yay_vote_count} yay votes and {nay_vote_count} nay votes on same line')
                    elif j == 13:
                        if verbose: print("Parsing Present Not Voting After Vote line on same line")
            
            if not matched:
                # Parsing bill name containing text
                if not bill_name_parsed:
                    bill_name = ''
                    current_line = line
                    invalid_vals = ['Y', 'N', 'X', 'P']
                    line_counter = i


                    while line_counter < len(lines) and current_line.strip() not in invalid_vals:
                        # Check if current line matches a vote pattern - if so, stop
                        vote_match = re.match(r'^(?P<VoteValue>[YNPXynpx]{1})$', current_line.strip())
                        if vote_match:
                            break
                            
                        if verbose: print(f'Appending {current_line} to bill name')
                        bill_name += current_line
                        line_counter += 1
                        
                        # Check bounds before accessing next line
                        if line_counter >= len(lines):
                            break
                        current_line = lines[line_counter]
                        
                    current_pdf_bill_data.append(bill_name)
                    bill_name_parsed = True
                    if verbose: print(f'Parsed bill name: {bill_name[:50]}...')

        # Switch around value order current_pdf_bill_data
        #bill_id, court_number	bill_name	rollcall_date	rollcall_time	rollcall_count_for_year	yea_vote_count	nay_vote_count	not_voting_count
        #0	185	20	135	5	01/14/2009	None	1	On adoption of an amendment (offered by Mr. Jo...
        # Only perform swaps if we have enough elements
        if len(current_pdf_bill_data) >= 9:
            current_pdf_bill_data[2], current_pdf_bill_data[8] = current_pdf_bill_data[8], current_pdf_bill_data[2] # move bill name in place, yay now at 8
            current_pdf_bill_data[3], current_pdf_bill_data[5] = current_pdf_bill_data[5], current_pdf_bill_data[3] # move rollcall_date in place, nay now at 5
            current_pdf_bill_data[4], current_pdf_bill_data[6] = current_pdf_bill_data[6], current_pdf_bill_data[4] # move rollcall_time in place, not voting now at 6
            current_pdf_bill_data[5], current_pdf_bill_data[7] = current_pdf_bill_data[7], current_pdf_bill_data[5] # move count_for_year in place, nay vote now at 7
            current_pdf_bill_data[6], current_pdf_bill_data[8] = current_pdf_bill_data[8], current_pdf_bill_data[6] # move yay_vote into place, rest should be in place now
                                                                      
        all_pdf_bills.append(current_pdf_bill_data)
        all_pdf_votes.extend(current_pdf_vote_data)
        bill_id += 1

    new_bill_id = bill_id
    print(f"Succesfully parsed file at path: {path}")
    if verbose:
        print(f'Found {len(all_pdf_bills)} bill records')
        print(f'Found {len(all_pdf_votes)} vote records')
    return all_pdf_bills, all_pdf_votes, new_bill_id


path = '../../data/HJRollCalls/2009/2009Combined_Vote2.pdf'
bill_data, vote_data, bill_id = parse_2009through2017_PDF(path, 0, 185, verbose=True)


PAGE #1
0: YEAS: 18         NAYS: 137
Matched with pattern ^YEAS: (?P<YeaVoteCount>\d+)\s+NAYS: (?P<NayVoteCount>\d+).?$
Found 18 yay votes and 137 nay votes on same line
1: N-V: 5
Matched with pattern ^N-V: (?P<NVVoteCount>\d+).?$
Found 5 NV votes
2: P=PRESENT        X=NOT VOTING      *=AFTER VOTE
Matched with pattern ^P=PRESENT        X=NOT VOTING      \*=AFTER VOTE$
Parsing Present Not Voting After Vote line on same line
3: Yea and Nay Supplement
Matched with pattern ^Yea and Nay Supplement$
Parsing Yay Nay supplement text line
Matched with pattern ^(?P<Name>\*?([A-Za-z][A-ZÀ-ÿa-z.,\ '’-]+?)(?: {,1}[\*À-ÿA-Za-z.,'’-]+)+).?$
Found person name but skipping because no associated vote found
4: JOURNAL OF THE HOUSE.
Matched with pattern ^JOURNAL OF THE HOUSE\.?$
Parsing JOURNAL OF THE HOUSE text line
Matched with pattern ^(?P<Name>\*?([A-Za-z][A-ZÀ-ÿa-z.,\ '’-]+?)(?: {,1}[\*À-ÿA-Za-z.,'’-]+)+).?$
Found person name but skipping because no associated vote found
5: Wednesday, January 14, 20

In [ ]:
# Test the parse_2009through2017_PDF function on all PDFs from 2009-2017
import traceback

# Get all PDF paths for years 2009-2017
test_years = range(2009, 2018)  # 2009 through 2017
test_paths = []
root_path = Path('../../data/HJRollCalls')

for year in test_years:
    year_path = root_path / str(year)
    if year_path.exists():
        pdf_files = list(year_path.glob('*.pdf'))
        test_paths.extend(pdf_files)
        print(f"Found {len(pdf_files)} PDFs for year {year}")

print(f"\nTotal PDFs to test: {len(test_paths)}\n")

# Test parsing on all files
bill_id = 0
all_bill_data = []
all_vote_data = []
failed_files = []
successful_files = []

for path in tqdm(test_paths, desc="Parsing PDFs"):
    try:
        # Get court number for the year
        file_year = int(path.parent.name)
        court_number = year_to_court_number[file_year]
        
        # Parse the PDF
        bill_data, vote_data, bill_id = parse_2009through2017_PDF(
            str(path), bill_id, court_number, verbose=False
        )
        
        all_bill_data.extend(bill_data)
        all_vote_data.extend(vote_data)
        successful_files.append(str(path))
        
    except Exception as e:
        failed_files.append((str(path), str(e)))
        print(f"\nERROR parsing {path.name}: {e}")
        if len(failed_files) <= 5:  # Only show traceback for first few errors
            traceback.print_exc()

# Print summary
print(f"\n{'='*60}")
print(f"Parsing Summary:")
print(f"{'='*60}")
print(f"Total files processed: {len(test_paths)}")
print(f"Successful: {len(successful_files)}")
print(f"Failed: {len(failed_files)}")
print(f"Total bills parsed: {len(all_bill_data)}")
print(f"Total votes parsed: {len(all_vote_data)}")

if failed_files:
    print(f"\nFailed files ({len(failed_files)}):")
    for file_path, error in failed_files[:10]:  # Show first 10 failures
        print(f"  - {Path(file_path).name}: {error[:100]}")
    if len(failed_files) > 10:
        print(f"  ... and {len(failed_files) - 10} more")


Found 274 PDFs for year 2009
Found 120 PDFs for year 2010
Found 184 PDFs for year 2011
Found 192 PDFs for year 2012
Found 279 PDFs for year 2013
Found 226 PDFs for year 2014
Found 179 PDFs for year 2015
Found 380 PDFs for year 2016
Found 305 PDFs for year 2017

Total PDFs to test: 2139



Parsing PDFs:   0%|          | 0/2139 [00:00<?, ?it/s]

In [9]:
# Create a dataframe from the parsed PDF data for bills
bill_df = pd.DataFrame(bill_data, columns=['bill_id', 'court_number', 'bill_name', 'rollcall_date', 'rollcall_time', 'rollcall_count_for_year', 'yea_vote_count', 'nay_vote_count', 'not_voting_count'])

bill_df.set_index('bill_id', inplace=True)
bill_df

,court_number,bill_name,rollcall_date,rollcall_time,rollcall_count_for_year,yea_vote_count,nay_vote_count,not_voting_count
bill_id,,,,,,,,
0,185,On adoption of an amendment (offered by Mr. Jo...,01/14/2009,None,2,18,137,5


In [10]:
# Create a dataframe from parsed PDF data about votes
vote_df = pd.DataFrame(vote_data, columns=['court_number', 'rollcall_count_for_year', 'voter_name', 'vote_value'])
vote_df

,court_number,rollcall_count_for_year,voter_name,vote_value
0,185,2,Mr Speaker,N
1,185,2,Ehrlich,N
2,185,2,Madden,N
3,185,2,Speliotis,N
4,185,2,Aguiar,N
...,...,...,...,...
155,185,2,Smola,Y
156,185,2,Dykema,N
157,185,2,L Italien,N
158,185,2,Smizik,N


In [15]:
def parse_Post2017_PDF(path, bill_id, court_number, verbose=False):

    doc = pymupdf.open(path)
    all_pdf_bills = []
    all_pdf_votes = []

    file_year = int(str(path)[23:27])

    for page in doc:
        text = page.get_text()
        lines = text.split('\n')

        current_pdf_bill_data = []
        current_pdf_bill_data.append(bill_id)
        current_pdf_bill_data.append(court_number)
        rc_count_for_year = 0

        current_pdf_vote_data = []
        nay_vote_found = False
        end_found = False
        for i, line in enumerate(lines):
            if verbose: print(f'{i}: {line}')

            # Skip parsing if blank line
            if line.isspace():
                if verbose: print("Found blank line")
                continue

            if i == 7 and file_year not in years_to_except: # parse full title line individually for ease
                current_pdf_bill_data.insert(2, line.strip())
            elif file_year in years_to_except and i == 1:
                current_pdf_bill_data.insert(2, line.strip())

            # match = re.search(title_regex, line)
            matched = False
            if not end_found:
                for j, pattern in enumerate(post2017_patterns_to_match):
                    match = re.match(pattern, line)

                    if match:
                        matched = True
                        if verbose: print(f'Matched with pattern {pattern}')

                        if j == 0:
                            # Extract vote date and time
                            rc_date = match.group('RCDate')
                            current_pdf_bill_data.append(rc_date)
                            rc_time = match.group('RCTime')
                            current_pdf_bill_data.append(rc_time)
                            if verbose: print(f"Found date: {rc_date} and time: {rc_time}")
                        elif j == 1:
                            # Do nothing, just the title
                            if verbose: print("Parsing title line")
                            break
                        elif j == 2:
                            # Get vote count and type of vote since they appear each on a new line
                            rc_vote_count = match.group('VoteCount')
                            current_pdf_bill_data.append(rc_vote_count)

                            rc_vote_type = match.group('VoteType')

                            if verbose: print(f'Found {rc_vote_count} {rc_vote_type} votes')
                        elif j == 3:
                            # Parse RC Vote number
                            rc_count_for_year = match.group('RCNumberForYear')
                            current_pdf_bill_data.insert(3, rc_count_for_year) # Insert at index so it matches format of other PDF styles


                            if verbose: print(f'Found No. {rc_count_for_year} on current line')
                        elif j == 4:
                            # Parsing Yays and Nays text, do nothing
                            if verbose: print("Parsing \'Yays and Nays\'")
                            break
                        elif j == 5:
                            # Skip parsing this line if previous line was a Nay Vote
                            if nay_vote_found and file_year not in years_to_except:
                                nay_vote_found = False
                                continue

                            # Parsing Voter Names
                            name = match.group('Name')

                            # Get Next line from list of lines to parse name for
                            vote_match = re.match('^(?P<VoteValue>\D{1})$', lines[i+1])
                            vote = vote_match.group('VoteValue')

                            current_pdf_vote_data.append([court_number, rc_count_for_year, name, vote])

                            if verbose: print(f'Found vote {vote} from Rep. {name}')
                        elif j == 6:
                            # Parse vote value, if value is N for Nay, parse name as well since they appear in reverse order

                            # Get Next line from list of lines to parse name for
                            vote_match = re.match('^(?P<VoteValue>\D{1})$', lines[i])
                            vote = vote_match.group('VoteValue')

                            if vote == 'N' and file_year not in years_to_except:
                                name_match = re.match('^(?P<Name>Mr. Speaker|([A-Z-][À-ÿa-z.,\'’-]+?)(?: {,1}[\*À-ÿA-Za-z.,\'’-]+)+)$', lines[i+1])
                                name = name_match.group('Name')

                                current_pdf_vote_data.append([court_number, rc_count_for_year, name, vote])
                                nay_vote_found = True
                                if verbose: print("Found Nay vote, will skip parsing next name")
                            else:
                                continue
                        elif j == 7:
                            if verbose: print("Parsing end line")
                            end_found = True
                            break
                if not matched:
                    Exception(f"Failed to match regex on line {i} : {line}")

        current_pdf_bill_data[4], current_pdf_bill_data[5] = current_pdf_bill_data[5], current_pdf_bill_data[4]
        all_pdf_bills.append(current_pdf_bill_data)
        all_pdf_votes.extend(current_pdf_vote_data)
        bill_id += 1

    new_bill_id = bill_id
    print(f"Succesfully parsed file at path: {path}")
    if verbose:
        print(f'Found {len(all_pdf_votes)} voting records')
        print(f'Found {len(all_pdf_bills)} bill records')
    return all_pdf_bills, all_pdf_votes, new_bill_id

### Testing on a Single PDF document

In [55]:
bill_data = []
vote_data = []
bill_id = 0
path = "../../data/HJRollCalls/2003/hj01082003.pdf"

# Call defined function to get data for current pdf
pdf_bill_data, pdf_vote_data, bill_id = parse_Pre2009_PDF(path, bill_id, 182)

bill_data.extend(pdf_bill_data)
vote_data.extend(pdf_vote_data)

Succesfully parsed file at path: ../../data/HJRollCalls/2003/hj01082003.pdf


In [56]:
bill_data

[[0, 182, 'QUORUM', '01/08/2003', '01:44 PM', '1', '151', '0', '9'],
 [1,
  182,
  'H2003  On motion to       Postpone',
  '01/08/2003',
  '02:01 PM',
  '2',
  '45',
  '107',
  '8'],
 [2,
  182,
  'H2003  On Amendment',
  '01/08/2003',
  '02:20 PM',
  '3',
  '41',
  '110',
  '9'],
 [3,
  182,
  'H2003  On Amendment',
  '01/08/2003',
  '02:45 PM',
  '4',
  '44',
  '107',
  '9'],
 [4,
  182,
  'H2003  On Amendment',
  '01/08/2003',
  '04:03 PM',
  '5',
  '28',
  '119',
  '13'],
 [5,
  182,
  'H2003  On Amendment',
  '01/08/2003',
  '04:34 PM',
  '6',
  '22',
  '128',
  '10']]

In [57]:
vote_data

[[182, '1', 'Mr. Speaker', 'Y'],
 [182, '1', 'Fresolo', 'Y'],
 [182, '1', 'Murphy, K.J.', 'Y'],
 [182, '1', 'Swan', 'Y'],
 [182, '1', 'Asselin', 'Y'],
 [182, '1', 'Galvin', 'Y'],
 [182, '1', 'Nangle', 'Y'],
 [182, '1', 'Teahan', 'Y'],
 [182, '1', 'Atkins', 'Y'],
 [182, '1', 'Garry', 'Y'],
 [182, '1', 'Naughton', 'Y'],
 [182, '1', 'Timilty', 'Y'],
 [182, '1', 'Atsalis', 'Y'],
 [182, '1', 'Gobi', 'X'],
 [182, '1', 'Nyman', 'Y'],
 [182, '1', 'Tobin', 'Y'],
 [182, '1', 'Ayers', 'Y'],
 [182, '1', 'Goguen', 'Y'],
 [182, '1', "O'Brien", 'Y'],
 [182, '1', 'Toomey', 'Y'],
 [182, '1', 'Balser', 'Y'],
 [182, '1', 'Golden B.P.', 'Y'],
 [182, '1', "O'Flaherty", 'Y'],
 [182, '1', 'Torrisi', 'Y'],
 [182, '1', 'Binienda', 'Y'],
 [182, '1', 'Golden T.A.', 'Y'],
 [182, '1', 'Owens-Hicks', 'Y'],
 [182, '1', 'Travis', 'Y'],
 [182, '1', 'Blumer', 'Y'],
 [182, '1', 'Grant', 'Y'],
 [182, '1', 'Parente', 'Y'],
 [182, '1', 'Turkington', 'Y'],
 [182, '1', 'Bosley', 'Y'],
 [182, '1', 'Greene', 'Y'],
 [182, '1', 

In [11]:
# Create a dataframe from the parsed PDF data for bills
bill_df = pd.DataFrame(bill_data, columns=['bill_id', 'court_number', 'bill_name', 'rollcall_date', 'rollcall_time', 'rollcall_count_for_year', 'yea_vote_count', 'nay_vote_count', 'not_voting_count'])

bill_df.set_index('bill_id', inplace=True)
bill_df

,court_number,bill_name,rollcall_date,rollcall_time,rollcall_count_for_year,yea_vote_count,nay_vote_count,not_voting_count
bill_id,,,,,,,,
0,185,On adoption of an amendment (offered by Mr. Jo...,01/14/2009,None,2,18,137,5


In [12]:
# Create a dataframe from parsed PDF data about votes
vote_df = pd.DataFrame(vote_data, columns=['court_number', 'rollcall_count_for_year', 'voter_name', 'vote_value'])
vote_df

,court_number,rollcall_count_for_year,voter_name,vote_value
0,185,2,Mr Speaker,N
1,185,2,Ehrlich,N
2,185,2,Madden,N
3,185,2,Speliotis,N
4,185,2,Aguiar,N
...,...,...,...,...
155,185,2,Smola,Y
156,185,2,Dykema,N
157,185,2,L Italien,N
158,185,2,Smizik,N


## Batch Processing all PDFs

Now that we have a function for scraping the data from all pages of a single pdf, we can loop through all of the pdfs that are stored on disk in the data and create the full dataset

In [13]:
# Get path to all pdfs stored in HJRollCall folder
all_paths = []
root_path = Path('../../data/HJRollCalls')

for path in root_path.glob('**/*.pdf'):
    all_paths.append(path)

print(len(all_paths))

2510



### Parse all PDFs

This block is the core part of this notebook that actually iterates over the whole folder of PDFs and parses them.

Currently, this function is set to parse roll call votes that are either < 2009 or 2017 >. This is because PDFs from the years 2009 - 2017 are done is a specific style where the actual votes are stored as an image embedded in the PDF, not text, and require some special attention. In the worst case, this data may need to be manually scraped but future experiementation will be done to see if we can automate this process.

Each parse_####_PDF() function call takes 3 variables, path = the file path to parse, bill_id = the iterator of all bills parsed (used for reference in the database table), and an optional named parameter verbose, which is set to False by default. When verbose is set to True, the function will output debugging messages in case something goes wrong during the parsing process.

In [16]:
bill_id = -1
bill_data = []
vote_data = []

for path in all_paths:

    # If path includes the year <2009, skip for now
    file_year = int(str(path)[23:27])
    court_number = years_to_court_number[file_year]

    print(f'Attempting to parsing file at path: {path}')
    if file_year < 2009:
        if bill_id < 0: bill_id+1
        current_bill_data, current_vote_data, bill_id = parse_Pre2009_PDF(path, bill_id+1, court_number)
    # Skip files from 2009 - 2017 for now since they will have more things to deal with for parsing
    elif file_year >= 2009 and file_year <= 2017:
        if bill_id < 0: bill_id+1
        print(f'Skipping file from year: {file_year}')
        continue
    elif file_year > 2017:
        if bill_id < 0: bill_id+1
        current_bill_data, current_vote_data, bill_id = parse_Post2017_PDF(path, bill_id+1, court_number)
    else:
        print(f"File in unexpected year attempted to be parsed: {path, file_year}")
        break

    bill_data.extend(current_bill_data)
    vote_data.extend(current_vote_data)

Attempting to parsing file at path: ..\..\data\HJRollCalls\2003\hj01082003.pdf
Succesfully parsed file at path: ..\..\data\HJRollCalls\2003\hj01082003.pdf
Attempting to parsing file at path: ..\..\data\HJRollCalls\2003\hj01092003.pdf
Succesfully parsed file at path: ..\..\data\HJRollCalls\2003\hj01092003.pdf
Attempting to parsing file at path: ..\..\data\HJRollCalls\2003\hj01142003.pdf
Succesfully parsed file at path: ..\..\data\HJRollCalls\2003\hj01142003.pdf
Attempting to parsing file at path: ..\..\data\HJRollCalls\2003\hj01222003.pdf
Succesfully parsed file at path: ..\..\data\HJRollCalls\2003\hj01222003.pdf
Attempting to parsing file at path: ..\..\data\HJRollCalls\2003\hj02032003.pdf
Succesfully parsed file at path: ..\..\data\HJRollCalls\2003\hj02032003.pdf
Attempting to parsing file at path: ..\..\data\HJRollCalls\2003\hj02052003.pdf
Succesfully parsed file at path: ..\..\data\HJRollCalls\2003\hj02052003.pdf
Attempting to parsing file at path: ..\..\data\HJRollCalls\2003\hj0320

### Creating and writing DataFrames from parsed data then converting to CSV

In [17]:
# Create a dataframe from the parsed PDF data for bills
bill_df = pd.DataFrame(bill_data, columns=['bill_id', 'court_number', 'bill_name', 'rollcall_date', 'rollcall_time', 'rollcall_count_for_year', 'yea_vote_count', 'nay_vote_count', 'not_voting_count'])

bill_df.set_index('bill_id', inplace=True)
bill_df

,court_number,bill_name,rollcall_date,rollcall_time,rollcall_count_for_year,yea_vote_count,nay_vote_count,not_voting_count
bill_id,,,,,,,,
0,182,QUORUM,01/08/2003,01:44 PM,1,151,0,9
1,182,H2003 On motion to Postpone,01/08/2003,02:01 PM,2,45,107,8
2,182,H2003 On Amendment,01/08/2003,02:20 PM,3,41,110,9
3,182,H2003 On Amendment,01/08/2003,02:45 PM,4,44,107,9
4,182,H2003 On Amendment,01/08/2003,04:03 PM,5,28,119,13
...,...,...,...,...,...,...,...,...
3567,194,H. 4240 Shall item 7004-0109 (Shelter Workforc...,84,02:13 PM,10/08/2025,131,24,5
3568,194,H. 4240 Shall item 7004-3036 (Housing Services...,85,02:17 PM,10/08/2025,131,24,5
3569,194,H. 4240 Shall item 7007-0150 (Regional Economi...,86,02:23 PM,10/08/2025,153,2,5


In [18]:
# Create a dataframe from parsed PDF data about votes
vote_df = pd.DataFrame(vote_data, columns=['court_number', 'rollcall_count_for_year', 'voter_name', 'vote_value'])
vote_df

,court_number,rollcall_count_for_year,voter_name,vote_value
0,182,1,Mr. Speaker,Y
1,182,1,Fresolo,Y
2,182,1,"Murphy, K.J.",Y
3,182,1,Swan,Y
4,182,1,Asselin,Y
...,...,...,...,...
506579,194,88,Xiarhos,Y
506580,194,88,Field,X
506581,194,88,MacGregor,Y
506582,194,88,Sousa,Y


The parsed output can be written to csv files for future cleaning and processing

In [66]:
bill_df.to_csv('../../data/parsedcsv/hjbills_raw.tsv', index=True, index_label=bill_df.index.name, sep='\t')
vote_df.to_csv('../../data/parsedcsv/hjrollcallvotes_raw.tsv', index=False, sep='\t')

Read the files from CSV to test that they wrote properly

In [68]:
bill_df_from_csv = pd.read_csv('../../data/parsedcsv/hjbills_raw.tsv', sep='\t')

bill_df_from_csv.head()

,bill_id,bill_name,rollcall_date,rollcall_time,rollcall_count_for_year,yea_vote_count,nay_vote_count,not_voting_count
0,0,QUORUM,06/08/2022,01:36 PM,194,155.0,0.0,0.0
1,1,H. 4805 shall this pass notwithstanding Objection,06/08/2022,01:50 PM,195,119.0,36.0,0.0
2,3,QUORUM,07/13/2022,01:32 PM,212,154.0,0.0,0.0
3,4,"H. 5007 Consolidated Amendment ""A""",07/13/2022,05:49 PM,213,154.0,0.0,0.0
4,6,QUORUM,02/16/2022,01:50 PM,144,156.0,0.0,0.0


In [70]:
vote_df_from_csv = pd.read_csv('../../data/parsedcsv/hjrollcallvotes_raw.tsv', sep='\t')
vote_df_from_csv.head()

,bill_id,voter_name,vote_value
0,0,Mr. Speaker,Y
1,0,Farley-Bouvier,Y
2,0,Mahoney,Y
3,0,Turco,Y
4,0,Hogan,Y


## Debugging individual PDFs

The following section is used for debugging specific documents that did not work. It is functionally identical to the section ### Testing on a single document, but is kept down here for reference for the work process, and less for presentation purposes.

In [8]:
path = '../../data/HJRollCalls/2009/2009combined.pdf'
doc = pymupdf.open(path)

for page in doc:
    print("NEW DOC STARTING \n\n")
    text = page.get_text()
    lines = text.split('\n')

    for line in lines:
        print(line)


NEW DOC STARTING 


YEAS: 20 
NAYS: 135
N-V: 5
P=PRESENT 
X=NOT VOTING 
*=AFTER VOTE
Yea and Nay Supplement
JOURNAL OF THE HOUSE.
Wednesday, January 14, 2009.
Yea and Nay No. 1
On adoption of an amendment (offered by Mr. Jones of North
Reading, et al) to the Bill expanding the Governor’s authority to address
deficiencies in revenue (House, No. 98). 
N 
Mr . Speaker 
N 
Ehrlich 
N 
Madden 
N 
Spel.iotis 
N 
Aguiar 
N 
Fagan 
N 
Mal.ia 
N 
Spel.l.ane 
N 
Al.icea 
X 
Fall.on 
N 
Mariano 
N 
Speranza 
N 
Al.l.en 
N 
Fal.zone 
N 
McCarthy 
N 
Spil.iotis 
N 
Arciero 
N 
Fennel.l. 
N 
McMurtry 
N 
Stanl.ey, H. L. 
N 
Ashe 
y 
Fernandes 
N 
Micel.i 
y 
Stanl.ey, T. M. 
N 
Atkins 
N 
Ferrante 
N 
Moran 
N 
St. Fleur 
N 
Atsalis 
N 
Finegol.d 
N 
Murphy, C.A . 
N 
Story 
N 
Ayers 
X 
Flynn 
X 
Murphy, J. 
N 
Straus 
y 
Balser 
N 
Forry 
N 
Murphy, K.J . 
N 
Sullivan 
N 
Basile 
N 
*Fox 
N 
Nangle 
N 
Swan 
N 
Benson 
y 
Fresol.o 
N 
Naughton 
N 
Timil.ty 
N 
Binienda 
N 
Galvin 
N 
Nyman 
N 
Tob

In [ ]:
bill_data, vote_data, bill_id = parse_2009through2017_PDF(path, 0, 185, verbose=True)

In [68]:
bill_data

[[0,
  185,
  'On adoption of an amendment (offered by Mr. Jones of NorthReading, et al) to the Bill expanding the Governor’s authority to addressdeficiencies in revenue (House, No. 98). ',
  '01/14/2009',
  None,
  1,
  20,
  135,
  5]]

In [69]:
# Create a dataframe from the parsed PDF data for bills
bill_df = pd.DataFrame(bill_data, columns=['bill_id', 'court_number', 'bill_name', 'rollcall_date', 'rollcall_time', 'rollcall_count_for_year', 'yea_vote_count', 'nay_vote_count', 'not_voting_count'])

bill_df.set_index('bill_id', inplace=True)
bill_df

,court_number,bill_name,rollcall_date,rollcall_time,rollcall_count_for_year,yea_vote_count,nay_vote_count,not_voting_count
bill_id,,,,,,,,
0,185,On adoption of an amendment (offered by Mr. Jo...,01/14/2009,None,1,20,135,5


In [72]:
# Create a dataframe from parsed PDF data about votes
vote_df = pd.DataFrame(vote_data, columns=['bill_id', 'voter_name', 'vote_value'])
vote_df[vote_df['voter_name'].str.contains('\*')]

,bill_id,voter_name,vote_value
41,0,*Fox,N
